# Exploring Models: Under the Hood of Transformers

Notes and experiments with the lower-level side of the `transformers` library --
working directly with `AutoModelForCausalLM` instead of the high-level
`pipeline()` API. This is where I actually load real model weights, look at the
architecture, and generate text token by token.

**Goal for this notebook:** get past the "it just works" feeling of pipelines
and actually see what's happening -- load a real Llama model in 4-bit
quantization, inspect its layers, generate a response manually, then repeat the
same pattern across a handful of different open models (Phi, Gemma, Qwen,
DeepSeek) to compare them.

This runs on a low-cost or free Colab T4 GPU.


## Reminder to self: the misleading CUDA error

If I see an error like:

> `Runtime error: CUDA is required but not available for bitsandbytes...`

that's misleading -- it's not really a package version issue. It usually means
Colab swapped out my runtime underneath me. Fix:

1. `Runtime` menu -> Disconnect and delete runtime
2. Reload the notebook fresh, `Edit` menu -> Clear All Outputs
3. Reconnect to a new T4 (top-right button)
4. Check "View resources" to confirm the GPU is actually attached
5. Re-run all cells from the top, starting with the pip installs


## Setup


In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers
# bitsandbytes is Hugging Face's quantization library -- lets me load large models in reduced precision


In [ ]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gc


### Signing in to Hugging Face

1. Free account at https://huggingface.co, Settings -> new API token with the
   **write** permission box checked.
2. Click the key icon in the left sidebar, add secret: `HF_TOKEN = your_token`.
3. Run the cell below to log in.


In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)


## Getting access to Llama

Llama is gated -- Meta requires accepting their terms before the weights or
tokenizer can be downloaded. Steps:

1. Visit https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct (or the
   smaller https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct, which is
   faster to download but may need its own separate approval) and accept the
   terms, ideally using the same email as my Hugging Face account.
2. Approval usually lands within a couple of minutes, and covers the whole 3.1
   family of models once granted.

If the model-loading cell later gives a 403 permissions error, worth checking:
- Am I actually logged in? (`login()` succeeding is a good sign)
- Does my API key have full read/write permissions?
- Does the model page above show I actually have access, near the top?


In [ ]:
# Instruct models, plus one reasoning model

# Llama 3.1 -- larger; should already be approved from the step above
LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Llama 3.2 is smaller but might need separate access approval
# LLAMA = "meta-llama/Llama-3.2-1B-Instruct"

PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"


In [ ]:
messages = [
    {"role": "user", "content": "Tell a joke for a room of Data Scientists"}
  ]


## Quantization: fitting a big model on a small GPU

An 8B-parameter model at full precision needs way more memory than a free-tier
T4 has. `bitsandbytes` lets me load the model in **4-bit precision** instead of
the usual 16/32-bit -- a huge memory reduction with a fairly small quality
trade-off.


In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,                     # Load and store model weights in 4-bit precision
    bnb_4bit_use_double_quant=True,        # Quantize the quantization parameters too, for extra memory savings
    bnb_4bit_compute_dtype=torch.bfloat16, # Use BF16 for the actual computations
    bnb_4bit_quant_type="nf4"              # NF4 quantization, tuned for neural network weight distributions
)


## Loading the Llama tokenizer and preparing the input

Same `apply_chat_template` idea from the tokenizers notebook, but this time
returning PyTorch tensors directly (`return_tensors="pt"`) and moving them onto
the GPU, since that's what `model.generate()` will expect.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(LLAMA)          # Load the tokenizer for the Llama model
tokenizer.pad_token = tokenizer.eos_token                  # Use EOS token as the padding token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")  # Convert chat messages to input tensors, move to GPU


In [ ]:
inputs


### Loading the actual model

`device_map="auto"` lets Hugging Face figure out how to place the model across
available devices (GPU here), and `quantization_config=quant_config` applies
the 4-bit setup from above.


In [ ]:
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)


Checking how much memory the quantized model is actually using -- useful for
seeing quantization's effect concretely, rather than just trusting it worked.


In [ ]:
memory = model.get_memory_footprint() / 1e6
print(f"Memory footprint: {memory:,.1f} MB")


## Looking under the hood at the Transformer architecture

Printing the `model` object shows the actual layer structure -- this is a
PyTorch neural network implementing the Transformer architecture (Google, 2017).

Things worth noticing in the printout:

- It's built from **layers**, stacked one after another.
- There's an **embedding** layer -- this turns token IDs into 4,096-dimensional
  vectors.
- There are then repeated groups of layers called **decoder layers** (32 of
  them for Llama 3.1). Each decoder layer bundles together: (a) a self-attention
  layer, (b) a multi-layer perceptron (MLP) layer, and (c) normalization layers.
- At the very end there's an **LM head** layer -- this is what actually produces
  the output (a probability distribution over the next token).
- The printout should also confirm the model has been quantized to 4-bit, since
  that's the config passed in above.


In [ ]:
# Print the model architecture and look through the layers

model


### Going even deeper (optional)

If I want to go further than just the layer summary, the actual PyTorch
implementation is open source in the Hugging Face `transformers` repo:
https://github.com/huggingface/transformers

For example, here's the Llama 4 modeling code:
https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama4/modeling_llama4.py

Not necessary to dig into this to be productive as an AI engineer -- most of
the day-to-day work is selecting, optimizing, fine-tuning, and applying
existing models rather than implementing a Transformer from scratch. But it's a
genuinely interesting rabbit hole if curious.


## Generating text the manual way

Instead of a pipeline handling everything, calling `model.generate()` directly
returns raw output token IDs -- not readable text yet.


In [ ]:
outputs = model.generate(**inputs, max_new_tokens=80)
outputs[0]


That's just a tensor of token IDs -- not very useful on its own. Decoding it
back through the tokenizer gives the actual generated text.


In [ ]:
tokenizer.decode(outputs[0])


### Freeing GPU memory

Before loading the next model, explicitly deleting the current one and clearing
the CUDA cache. Colab's "Show Resources" panel might not update instantly, but
the memory does become available for whatever gets loaded next.


In [ ]:
del model, inputs, tokenizer, outputs
gc.collect()
torch.cuda.empty_cache()


## Streaming output, and wrapping this into a reusable function

Two changes worth understanding before the next block:

**Streaming:** `TextStreamer` prints tokens as they're generated instead of
waiting for the whole response, similar in spirit to the streaming chatbot
callback from earlier notebooks. It replaces:

```python
outputs = model.generate(inputs, max_new_tokens=80)
```

with:

```python
streamer = TextStreamer(tokenizer)
outputs = model.generate(inputs, max_new_tokens=80, streamer=streamer)
```

**`add_generation_prompt=True`:** added to the chat template call. This makes
sure the model actually generates a *reply* to the prompt, rather than just
predicting what comes next after the user's message (which, without this flag,
could just continue the user's sentence instead of answering it). Worth trying
with this set to `False` at some point to see the difference directly --
more detail here:
https://huggingface.co/docs/transformers/main/en/chat_templating#what-are-generation-prompts

Since I'll want to repeat this whole load-tokenizer -> load-model -> generate
pattern across several different models, wrapping it in a function saves a lot
of repetition.


In [ ]:
def generate(model_path_or_name, messages, quant=True, max_new_tokens=80):
  tokenizer = AutoTokenizer.from_pretrained(model_path_or_name)
  tokenizer.pad_token = tokenizer.eos_token

  # Get the BatchEncoding object containing input_ids and attention_mask
  encoded_chat = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True)

  # Extract individual tensors and move them to the CUDA device
  input_ids = encoded_chat['input_ids'].to(device="cuda")
  attention_mask = encoded_chat['attention_mask'].to(device="cuda")

  streamer = TextStreamer(tokenizer)
  if quant:
    model = AutoModelForCausalLM.from_pretrained(model_path_or_name, quantization_config=quant_config).to("cuda")
  else:
    model = AutoModelForCausalLM.from_pretrained(model_path_or_name).to("cuda")

  outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, streamer=streamer)


## Trying this on other open models

Now reusing the same `generate` function across a handful of different models,
to compare them on the same prompt. First, Phi-4 from Microsoft.


In [ ]:
generate(PHI, messages)


### Gemma from Google

Gemma is also gated -- need to accept Google's terms on the model page before
it'll download: https://huggingface.co/google/gemma-3-270m-it

Loading unquantized here (`quant=False`) since it's small enough to fit
comfortably without 4-bit quantization.


In [ ]:
messages = [
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
  ]
generate(GEMMA, messages, quant=False)


### Qwen from Alibaba Cloud


In [ ]:
generate(QWEN, messages)


### DeepSeek -- a reasoning model

This one's a distilled reasoning model, so giving it more headroom
(`max_new_tokens=500`) makes sense -- reasoning models often "think out loud"
before landing on a final answer, which takes more tokens than a direct
response.


In [ ]:
generate(DEEPSEEK, messages, quant=False, max_new_tokens=500)
